# PitchVision — GPU inference server (Colab)

Runs the **v9** engine (T-DEED backbone + PitchVision head) behind a small FastAPI server, and exposes it
through a **cloudflared quick tunnel** so the PitchVision web app on your laptop can send it clips.

| Step | Cell |
|---|---|
| 1 | Check the GPU (Runtime → Change runtime type → **T4 GPU**) |
| 2 | Settings |
| 3 | Fetch PitchVision |
| 4 | Fetch **T-DEED** (GPL-3.0, an external dependency — cloned here, never vendored into the repo) and its checkpoint |
| 5 | Install the server and cloudflared |
| 6 | **Start the server + tunnel** → prints the line to paste into the web app |
| 7 | Self-test through the tunnel |
| 8 | *(optional)* Analyse a **full match** with no length cap, for the pre-analysed library |
| 9 | Logs / stop |

**Every Colab session gets a new tunnel URL.** After re-running cell 6, run the printed
`npm run set-backend -- <url> <token>` in `web/` — the web app picks it up without a restart.

**Licences.** T-DEED is GPL-3.0; it is cloned at a pinned commit and runs as a separate process. The
SoccerNetBall checkpoint and PitchVision's v9 head were trained on SoccerNet data: **non-commercial research
only**. Keep this tunnel private (the server requires the token printed in cell 6).

**Speed.** About 0.6× real time on a T4: a 2-minute clip takes ~75 s, a full match about an hour.

In [ ]:
#@title 1 · GPU check
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'No GPU visible — set Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 2 · Settings { display-mode: "form" }
REPO_URL      = "https://github.com/brianodh/pitchvision.git"  #@param {type:"string"}
BRANCH        = "feat/v9-engine"                                 #@param {type:"string"}
# Private repo? Add a GitHub token as a Colab secret named GITHUB_TOKEN (key icon, left sidebar).

TDEED_URL     = "https://github.com/arturxe2/T-DEED.git"
TDEED_COMMIT  = "d953b4de8a2ad941c565e1683c5b07255cbe644e"   # pinned; bump deliberately

# T-DEED's SoccerNetBall_test checkpoint. Download it once from the link in T-DEED's README into your Drive,
# then point at it here. (Do NOT use SoccerNetBall_challenge1: it was trained on the test matches.)
TDEED_CKPT_ON_DRIVE = "/content/drive/MyDrive/T-DEED/SoccerNetBall_test/checkpoint_best.pt"  #@param {type:"string"}

MAX_MINUTES   = 10       #@param {type:"number"}
THUMBS        = "frame"  #@param ["frame", "player", "none"]
# "player" crops the person nearest frame centre with YOLOv8 (ultralytics, AGPL-3.0) — off by default.
CLOUDFLARED_VERSION = "2026.9.3"

WORK = "/content"
REPO = f"{WORK}/pitchvision"
TDEED = f"{WORK}/T-DEED"

In [ ]:
#@title 3 · Fetch PitchVision
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://x-access-token:{tok}@')
except Exception:
    pass
if not os.path.isdir(REPO):
    r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(url, REPO_URL)[-2000:])
else:
    print(subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], capture_output=True, text=True).stdout)
assert os.path.exists(f'{REPO}/backend/pv_server/app.py'), 'clone failed — check REPO_URL / BRANCH / GITHUB_TOKEN'
print('PitchVision at', subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout)

In [ ]:
#@title 4 · Fetch T-DEED (external, GPL-3.0) and its checkpoint
import os, shutil, subprocess, sys
if not os.path.isdir(TDEED):
    subprocess.run(['git', 'clone', TDEED_URL, TDEED], check=True)
subprocess.run(['git', '-C', TDEED, 'checkout', '--quiet', TDEED_COMMIT], check=True)

# Into THIS interpreter's environment — the feature-capture subprocess runs with sys.executable.
# (Don't install T-DEED's requirements.txt: it pins an old torch and would break Colab's.)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.3', 'tabulate', 'SoccerNet'], check=True)

cfg_dir = f'{TDEED}/config/SoccerNetBall'
if not os.path.exists(f'{cfg_dir}/SoccerNetBall_test.json'):
    shutil.copy(f'{cfg_dir}/SoccerNetBall_challenge1.json', f'{cfg_dir}/SoccerNetBall_test.json')

dst = f'{TDEED}/checkpoints/SoccerNetBall/SoccerNetBall_test/checkpoint_best.pt'
if not os.path.exists(dst):
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.exists(TDEED_CKPT_ON_DRIVE), f'checkpoint not found at {TDEED_CKPT_ON_DRIVE} — see cell 2'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(TDEED_CKPT_ON_DRIVE, dst)
print('T-DEED ready at', TDEED, '@', TDEED_COMMIT[:10], '| checkpoint', os.path.getsize(dst) // 2**20, 'MB')

In [ ]:
#@title 5 · Install the server and cloudflared
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'python-multipart'], check=True)
if THUMBS == 'player':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
CF = f'{WORK}/cloudflared'
if not os.path.exists(CF):
    subprocess.run(['wget', '-q', '-O', CF,
                    f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64'],
                   check=True)
    os.chmod(CF, 0o755)
print(subprocess.run([CF, '--version'], capture_output=True, text=True).stdout)
print(subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
#@title 6 · Start the server + tunnel
import os, re, secrets, subprocess, sys, time, urllib.request

for name in ('server_proc', 'tunnel_proc'):
    p = globals().get(name)
    if p and p.poll() is None:
        p.terminate(); p.wait(10)

TOKEN = globals().get('TOKEN') or secrets.token_urlsafe(24)
env = dict(os.environ,
           PV_ENGINE='v9', TDEED_ROOT=TDEED, PV_MODEL_ROOT=f'{REPO}/model', PV_TOKEN=TOKEN,
           PV_MAX_SECONDS=str(int(MAX_MINUTES * 60)), PV_THUMBS=THUMBS, PV_WORKDIR=f'{WORK}/pv_jobs')
server_log = open(f'{WORK}/server.log', 'w')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'pv_server.app:create_app', '--factory', '--host', '127.0.0.1', '--port', '8000'],
    cwd=f'{REPO}/backend', env=env, stdout=server_log, stderr=subprocess.STDOUT)

def health(base, timeout=5):
    req = urllib.request.Request(base + '/health', headers={'Authorization': f'Bearer {TOKEN}'})
    return urllib.request.urlopen(req, timeout=timeout).read().decode()

for _ in range(60):
    try:
        health('http://127.0.0.1:8000'); break
    except Exception:
        if server_proc.poll() is not None:
            raise RuntimeError('server exited:\n' + open(f'{WORK}/server.log').read()[-3000:])
        time.sleep(1)
else:
    raise RuntimeError('server did not start:\n' + open(f'{WORK}/server.log').read()[-3000:])

tunnel_log = open(f'{WORK}/tunnel.log', 'w')
tunnel_proc = subprocess.Popen([f'{WORK}/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                               stdout=tunnel_log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(f'{WORK}/tunnel.log').read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
assert URL, 'no tunnel URL yet:\n' + open(f'{WORK}/tunnel.log').read()[-2000:]

print('Server and tunnel running.\n')
print('In the web app folder (web/) on your laptop, run:\n')
print(f'    npm run set-backend -- {URL} {TOKEN}\n')
print('or put these in web/.env.local and restart `npm start`:\n')
print(f'    PV_BACKEND_URL={URL}\n    PV_BACKEND_TOKEN={TOKEN}\n')
print('Keep the token private: anyone with URL + token can use this GPU.')

In [ ]:
#@title 7 · Self-test through the tunnel (DNS for a new quick tunnel can take ~30 s)
import json, time
for i in range(12):
    try:
        print(json.dumps(json.loads(health(URL, timeout=10)), indent=1)); break
    except Exception as e:
        print('waiting for the tunnel…', type(e).__name__); time.sleep(5)

## 8 · Optional: analyse a full match for the pre-analysed library

The live API caps uploads at `MAX_MINUTES` because a full match takes about an hour on a T4. For the demo
library, run the engine directly on a match in your Drive (no cap), save `timeline.json` back to Drive, then on
your laptop:

```bash
python backend/scripts/build_demo.py --timeline timeline.json --video "<local copy of the same video>" \
    --title "Stoke City vs Huddersfield Town" --subtitle "EFL Championship · 1 Oct 2019"
```

The video file, frames and thumbnails stay on your machine — SoccerNet data may not be redistributed.

In [ ]:
#@title 8 · Full-match run (about an hour) { display-mode: "form" }
MATCH_VIDEO = "/content/drive/MyDrive/SoccerNet_Project/extracted/test/england_efl/2019-2020/2019-10-01 - Stoke City - Huddersfield Town/224p.mp4"  #@param {type:"string"}
OUT_ON_DRIVE = "/content/drive/MyDrive/pitchvision_out/stoke_huddersfield"  #@param {type:"string"}
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_ON_DRIVE, exist_ok=True)
env = dict(os.environ, TDEED_ROOT=TDEED)
subprocess.run([sys.executable, 'scripts/spot_video_v9.py', '--video', MATCH_VIDEO, '--out', OUT_ON_DRIVE],
               cwd=f'{REPO}/model', env=env, check=True)
print('timeline:', os.path.join(OUT_ON_DRIVE, 'timeline.json'))

In [ ]:
#@title 9 · Logs
print('--- server ---'); print(open(f'{WORK}/server.log').read()[-4000:])
print('--- tunnel ---'); print(open(f'{WORK}/tunnel.log').read()[-1500:])
print('server running:', server_proc.poll() is None, '| tunnel running:', tunnel_proc.poll() is None)

In [ ]:
#@title Stop server and tunnel
for p in (globals().get('tunnel_proc'), globals().get('server_proc')):
    if p and p.poll() is None:
        p.terminate()
print('stopped')